In [11]:
!pip install torch torchvision

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [12]:
import os
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from PIL import Image

# ==========================================
# DATASET PATH
# ==========================================

BASE_PATH = r"C:\Users\artec3d\Desktop\Datasets_raw\C-NMC_Leukemia"

TRAIN_PATH = os.path.join(
    BASE_PATH,
    "training_data",
    "fold_0"
)

# ==========================================
# CLASS LABELS
# ==========================================

CLASS_MAP = {
    "all": 0,
    "hem": 1
}

# ==========================================
# IMAGE TRANSFORMS
# ==========================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ==========================================
# CUSTOM DATASET
# ==========================================

class LeukemiaDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = root_dir
        self.transform = transform

        self.samples = []

        for class_name in CLASS_MAP:

            class_folder = os.path.join(root_dir, class_name)

            for img_path in Path(class_folder).glob("*"):

                self.samples.append(
                    (
                        str(img_path),
                        CLASS_MAP[class_name]
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        sample = {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "path": img_path
        }

        return sample

# ==========================================
# CREATE DATASET
# ==========================================

train_dataset = LeukemiaDataset(
    TRAIN_PATH,
    transform=transform
)

# ==========================================
# CREATE DATALOADER
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# ==========================================
# CHECK LOADING
# ==========================================

batch = next(iter(train_loader))

print("Image Shape :", batch["image"].shape)
print("Labels Shape:", batch["label"].shape)

print("Example Path:")
print(batch["path"][0])

Image Shape : torch.Size([32, 3, 224, 224])
Labels Shape: torch.Size([32])
Example Path:
C:\Users\artec3d\Desktop\Datasets_raw\C-NMC_Leukemia\training_data\fold_0\hem\UID_H24_31_13_hem.bmp


In [20]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from PIL import Image

# ==========================================
# DEVICE
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

# ==========================================
# DATASET PATH
# ==========================================

BASE_PATH = r"C:\Users\artec3d\Desktop\Datasets_raw\C-NMC_Leukemia"

TRAIN_PATH = os.path.join(
    BASE_PATH,
    "training_data",
    "fold_0"
)

# ==========================================
# CLASS LABELS
# ==========================================

CLASS_MAP = {
    "all": 0,
    "hem": 1
}

# ==========================================
# IMAGE TRANSFORMS
# ==========================================

transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor()

])

# ==========================================
# CUSTOM DATASET
# ==========================================

class LeukemiaDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = root_dir
        self.transform = transform

        self.samples = []

        for class_name in CLASS_MAP:

            class_folder = os.path.join(root_dir, class_name)

            for img_path in Path(class_folder).glob("*"):

                self.samples.append(
                    (
                        str(img_path),
                        CLASS_MAP[class_name]
                    )
                )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:

            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# ==========================================
# CREATE DATASET
# ==========================================

train_dataset = LeukemiaDataset(
    TRAIN_PATH,
    transform=transform
)

# ==========================================
# CREATE DATALOADER
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# ==========================================
# CHECK DATA LOADING
# ==========================================

images, labels = next(iter(train_loader))

print("Image Shape :", images.shape)
print("Labels Shape:", labels.shape)

print("Total Loaded Images :", len(train_dataset))

# ==========================================
# LOAD PRETRAINED RESNET18
# ==========================================

model = models.resnet18(pretrained=True)

# ==========================================
# MODIFY FINAL LAYER
# ==========================================

num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 2)

model = model.to(device)

# ==========================================
# LOSS FUNCTION + OPTIMIZER
# ==========================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ==========================================
# TRAINING LOOP
# ==========================================

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)

        labels = labels.to(device)

        # ==================================
        # FORWARD PASS
        # ==================================

        outputs = model(images)

        loss = criterion(outputs, labels)

        # ==================================
        # BACKPROPAGATION
        # ==================================

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # ==================================
        # CALCULATE ACCURACY
        # ==================================

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    # ======================================
    # EPOCH RESULTS
    # ======================================

    epoch_loss = running_loss / len(train_loader)

    accuracy = 100 * correct / total

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")

    print(f"Loss     : {epoch_loss:.4f}")

    print(f"Accuracy : {accuracy:.2f}%")

Using Device: cuda
Image Shape : torch.Size([32, 3, 224, 224])
Labels Shape: torch.Size([32])
Total Loaded Images : 3527

Epoch [1/5]
Loss     : 0.3842
Accuracy : 85.54%

Epoch [2/5]
Loss     : 0.2610
Accuracy : 89.65%

Epoch [3/5]
Loss     : 0.2598
Accuracy : 90.05%

Epoch [4/5]
Loss     : 0.2214
Accuracy : 91.69%

Epoch [5/5]
Loss     : 0.1823
Accuracy : 92.80%


In [13]:
from collections import Counter

labels = [sample[1] for sample in train_dataset.samples]

class_counts = Counter(labels)

print("Class-wise Image Count:\n")

for class_name, label in CLASS_MAP.items():

    print(f"{class_name} : {class_counts[label]}")
    

Class-wise Image Count:

all : 2397
hem : 1130


In [18]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ==========================================
# 1. Configuration & Hyperparameters
# ==========================================
CSV_PATH = r"C:\Users\artec3d\Desktop\Datasets_raw\enhanced_fever_medicine_recommendation.csv"

BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001

# Define our feature types based on the CSV
NUMERICAL_COLS = ['Temperature', 'Age', 'BMI', 'Humidity', 'AQI', 'Heart_Rate']
CATEGORICAL_COLS = [
    'Fever_Severity', 'Gender', 'Headache', 'Body_Ache', 'Fatigue', 
    'Chronic_Conditions', 'Allergies', 'Smoking_History', 'Alcohol_Consumption', 
    'Physical_Activity', 'Diet_Type', 'Blood_Pressure', 'Previous_Medication'
]
TARGET_COL = 'Recommended_Medication'

# ==========================================
# 2. Data Loading & Preprocessing
# ==========================================
print("Loading and preprocessing data...")
df = pd.read_csv(CSV_PATH)

# Encode Target
target_encoder = LabelEncoder()
df[TARGET_COL] = target_encoder.fit_transform(df[TARGET_COL])
num_classes = len(target_encoder.classes_)

# Scale Numerical Features
scaler = StandardScaler()
df[NUMERICAL_COLS] = scaler.fit_transform(df[NUMERICAL_COLS])

# Encode Categorical Features to Integers (for Embedding Layers)
cat_encoders = {}
cat_dims = [] # To store the number of unique values for each category

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    cat_encoders[col] = le
    cat_dims.append(len(le.classes_))

# ==========================================
# 3. PyTorch Dataset Definition
# ==========================================
class FeverDataset(Dataset):
    def __init__(self, dataframe):
        self.num_data = dataframe[NUMERICAL_COLS].values.astype(np.float32)
        self.cat_data = dataframe[CATEGORICAL_COLS].values.astype(np.int64)
        self.targets = dataframe[TARGET_COL].values.astype(np.int64)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.num_data[idx], self.cat_data[idx], self.targets[idx]

# Split data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = FeverDataset(train_df)
test_dataset = FeverDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. Multimodal-Ready Neural Network Model
# ==========================================
class TabularEmbeddingModel(nn.Module):
    def __init__(self, cat_dims, num_numerical_cols, num_classes):
        super(TabularEmbeddingModel, self).__init__()
        
        # Create an Embedding Layer for each categorical column
        # Rule of thumb for embedding size: min(50, num_categories // 2 + 1)
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=dim, embedding_dim=min(50, (dim + 1) // 2))
            for dim in cat_dims
        ])
        
        # Calculate total dimension after embeddings are concatenated
        total_emb_dim = sum(min(50, (dim + 1) // 2) for dim in cat_dims)
        
        # Total input dimension = embedded categories + numerical features
        input_dim = total_emb_dim + num_numerical_cols
        
        # This is the "Assembly Line" we talked about
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        
        # THIS is the layer that outputs our "Tabular Embedding" (e.g., 16 numbers)
        # In the future, we will extract the output of this layer and combine it with Images/Audio
        self.tabular_embedding_layer = nn.Linear(64, 16) 
        
        # Final Decision Layer (Only used when evaluating the tabular data alone)
        self.output_layer = nn.Linear(16, num_classes)

    def forward(self, num_x, cat_x):
        # 1. Pass categorical data through embedding layers
        emb_outputs = []
        for i, emb_layer in enumerate(self.embeddings):
            emb_outputs.append(emb_layer(cat_x[:, i]))
            
        # 2. Concatenate all embeddings together
        cat_embedded = torch.cat(emb_outputs, dim=1)
        
        # 3. Concatenate numerical data with the embedded categorical data
        x = torch.cat([num_x, cat_embedded], dim=1)
        
        # 4. Pass through hidden layers
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # 5. Extract Tabular Embedding! (The 16 numbers)
        tabular_embedding = self.tabular_embedding_layer(x)
        tabular_embedding = self.relu(tabular_embedding)
        
        # 6. Final prediction
        out = self.output_layer(tabular_embedding)
        return out

model = TabularEmbeddingModel(cat_dims, len(NUMERICAL_COLS), num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==========================================
# 5. Training Loop
# ==========================================
print("\nStarting Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for num_x, cat_x, labels in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(num_x, cat_x)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss/len(train_loader):.4f} - Accuracy: {epoch_acc:.2f}%")

print("\nModel trained successfully!")

# Save the model weights
torch.save(model.state_dict(), 'tabular_model_weights.pth')
print("Model saved to 'tabular_model_weights.pth'")
print("\nThis model is now fully prepared for multimodal integration. You can strip the 'output_layer' later to extract the 16-dimensional tabular embeddings!")



# 1. Instantiate the model architecture
model = TabularEmbeddingModel(cat_dims, len(NUMERICAL_COLS), num_classes)

# 2. Load the pre-trained weights you pushed to GitHub
model.load_state_dict(torch.load('tabular_model_weights.pth'))

# 3. Put the model in "Evaluation Mode" (Crucial step!)
model.eval()

print("Pre-trained weights loaded successfully! No training required.")

# Grab one batch of patients from the test data
num_x, cat_x, labels = next(iter(test_loader))

# Let's look at the very first patient in this batch
patient_num = num_x[0].unsqueeze(0)
patient_cat = cat_x[0].unsqueeze(0)

# Pass the patient through the hidden layers (but NOT the final output layer)
with torch.no_grad():
    # 1. Get categorical embeddings
    emb_outputs = [emb_layer(patient_cat[:, i]) for i, emb_layer in enumerate(model.embeddings)]
    cat_embedded = torch.cat(emb_outputs, dim=1)
   
    # 2. Mix with numerical data
    x = torch.cat([patient_num, cat_embedded], dim=1)
    x = model.fc1(x)
    x = model.relu(x)
   
    # 3. Extract the final 16-number tabular embedding!
    tabular_embedding = model.tabular_embedding_layer(x)
    tabular_embedding = model.relu(tabular_embedding)

print("Here is the 16-number embedding for this patient:")
print(tabular_embedding)



Loading and preprocessing data...

Starting Training...
Epoch 1/20 - Loss: 0.5387 - Accuracy: 79.88%
Epoch 2/20 - Loss: 0.3997 - Accuracy: 79.50%
Epoch 3/20 - Loss: 0.2677 - Accuracy: 86.00%
Epoch 4/20 - Loss: 0.1267 - Accuracy: 97.88%
Epoch 5/20 - Loss: 0.0492 - Accuracy: 100.00%
Epoch 6/20 - Loss: 0.0209 - Accuracy: 100.00%
Epoch 7/20 - Loss: 0.0095 - Accuracy: 100.00%
Epoch 8/20 - Loss: 0.0057 - Accuracy: 100.00%
Epoch 9/20 - Loss: 0.0042 - Accuracy: 100.00%
Epoch 10/20 - Loss: 0.0030 - Accuracy: 100.00%
Epoch 11/20 - Loss: 0.0033 - Accuracy: 99.88%
Epoch 12/20 - Loss: 0.0016 - Accuracy: 100.00%
Epoch 13/20 - Loss: 0.0011 - Accuracy: 100.00%
Epoch 14/20 - Loss: 0.0016 - Accuracy: 100.00%
Epoch 15/20 - Loss: 0.0007 - Accuracy: 100.00%
Epoch 16/20 - Loss: 0.0007 - Accuracy: 100.00%
Epoch 17/20 - Loss: 0.0009 - Accuracy: 100.00%
Epoch 18/20 - Loss: 0.0005 - Accuracy: 100.00%
Epoch 19/20 - Loss: 0.0004 - Accuracy: 100.00%
Epoch 20/20 - Loss: 0.0005 - Accuracy: 100.00%

Model trained suc